In [0]:
base = "/Volumes/workspace/default/training_files/"   # change this to your upload path

students    = spark.read.csv(base + "students_cleaned.csv",    header=True, inferSchema=True)
courses     = spark.read.csv(base + "courses_cleaned.csv",     header=True, inferSchema=True)
enrollments = spark.read.csv(base + "enrollments_cleaned.csv", header=True, inferSchema=True)
progress    = spark.read.csv(base + "progress_cleaned.csv",    header=True, inferSchema=True)

display(students.limit(5))
students.printSchema()

student_id,student_name,email,city,join_date
S001,Asha Raman,asha.raman@example.com,Chennai,2023-12-01
S002,Karthik Kumar,karthik.kumar@example.com,Coimbatore,2023-12-05
S003,Meena Iyer,meena.iyer@example.com,Madurai,2023-12-10
S005,Ravi Shankar,ravi.shankar@example.com,Chennai,2024-01-03
S006,Divya Lakshmi,divya.lakshmi@example.com,Salem,2024-01-05


root
 |-- student_id: string (nullable = true)
 |-- student_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- join_date: date (nullable = true)



In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col

enrollments = (enrollments
    .dropDuplicates(["student_id", "course_id"])
    .dropna(subset=["student_id", "course_id"])
    .withColumn("enrollment_date", F.to_date("enrollment_date")))

progress = (progress
    .withColumn("completion", F.col("completion").cast("double"))
    .fillna({"completion": 0}))

In [0]:
joined = (enrollments
    .join(students, "student_id", "left")
    .join(courses,  "course_id",  "left")
    .join(progress, ["student_id", "course_id"], "left"))

display(joined)

student_id,course_id,enrollment_id,student_name,course_name,enrollment_date,student_name,email,city,join_date,course_name,category,duration_hours,price,progress_id,enrollment_id,completion,last_accessed
S007,C104,E007,Suresh Babu,PySpark Essentials,2024-02-10,Suresh Babu,suresh.babu@example.com,Kumbakonam,2024-01-08,PySpark Essentials,Big Data,30.0,3499,P007,E007,75.0,2024-03-07
S001,C101,E001,Asha Raman,Python Basics,2024-01-10,Asha Raman,asha.raman@example.com,Chennai,2023-12-01,Python Basics,Programming,20.0,1999,P001,E001,85.0,2024-03-01
S013,C105,E013,Manoj Kumar,Azure Databricks,2024-03-05,Manoj Kumar,manoj.kumar@example.com,Chennai,2024-02-03,Azure Databricks,Cloud,35.0,3999,P013,E013,40.0,2024-03-11
S001,C102,E016,Asha Raman,SQL Fundamentals,2024-03-10,Asha Raman,asha.raman@example.com,Chennai,2023-12-01,SQL Fundamentals,Database,25.0,2499,P016,E016,30.0,2024-03-14
S002,C101,E002,Karthik Kumar,Python Basics,2024-01-12,Karthik Kumar,karthik.kumar@example.com,Coimbatore,2023-12-05,Python Basics,Programming,20.0,1999,P002,E002,92.5,2024-03-02
S009,C101,E009,Arun Prakash,Python Basics,2024-02-14,Arun Prakash,arun.prakash@example.com,Chennai,2024-01-15,Python Basics,Programming,20.0,1999,P009,E009,100.0,2024-03-09
S005,C103,E005,Ravi Shankar,Data Analysis with Pandas,2024-02-01,Ravi Shankar,ravi.shankar@example.com,Chennai,2024-01-03,Data Analysis with Pandas,Data Analysis,27.5,2999,P005,E005,0.0,2024-03-05
S011,C103,E011,Vignesh Raj,Data Analysis with Pandas,2024-02-20,Vignesh Raj,vignesh.raj@example.com,Madurai,2024-01-20,Data Analysis with Pandas,Data Analysis,27.5,2999,null,null,null,null
S017,C104,E019,Gayathri Mohan,PySpark Essentials,2024-03-18,Gayathri Mohan,gayathri.mohan@example.com,Kumbakonam,2024-02-12,PySpark Essentials,Big Data,30.0,3499,P019,E019,88.0,2024-03-17
S012,C104,E012,Lavanya Devi,PySpark Essentials,2024-03-01,Lavanya Devi,lavanya.devi@example.com,Coimbatore,2024-02-01,PySpark Essentials,Big Data,30.0,3499,null,null,null,null


In [0]:
final_df = (joined
    .select(
        enrollments["student_name"],
        enrollments["course_name"].alias("course"),
        F.col("enrollment_date"),
        F.col("completion").alias("progress"))
    .orderBy("student_name", "course"))

display(final_df)
print("Rows:", final_df.count())

student_name,course,enrollment_date,progress
Arun Prakash,Python Basics,2024-02-14,100.0
Asha Raman,Python Basics,2024-01-10,85.0
Asha Raman,SQL Fundamentals,2024-03-10,30.0
Bala Murugan,Azure Databricks,2024-03-20,72.0
Gayathri Mohan,PySpark Essentials,2024-03-18,88.0
Harish Chandran,Azure Databricks,2024-03-08,95.0
Karthik Kumar,Data Analysis with Pandas,2024-03-12,65.0
Karthik Kumar,Python Basics,2024-01-12,92.5
Lavanya Devi,PySpark Essentials,2024-03-01,null
Manoj Kumar,Azure Databricks,2024-03-05,40.0


Rows: 15


In [0]:
# Delta table
final_df.write.format("delta").mode("overwrite").saveAsTable("student_course_progress")

# CSV for dashboards
(final_df.coalesce(1)
    .write.mode("overwrite")
    .option("header", True)
    .csv(base + "output/final_student_course_progress"))